# 🧬 04 · 基因结构与注释（ORF / 6 框翻译 / GFF）

> 有了基因序列，怎么**找到基因**？原核生物基因 = 由起始密码子开始、
> 到终止密码子结束的**开放阅读框（ORF）**。本节：遗传密码表、
> 6 框翻译手写、ORF 检测、GFF 注释文件解析。

## 📋 本节要解决的问题

- 标准遗传密码：64 密码子 → 20 氨基酸 + 3 终止，怎么查表？
- 6 框翻译是什么？为什么真核注释更复杂（内含子）？
- ORF 怎么检测？起始/终止密码子有哪些？
- GFF 注释文件什么格式？

## 1️⃣ 遗传密码表

64 个密码子中：61 个编码氨基酸，3 个终止（TAA/TAG/TGA），1 个起始兼甲硫氨酸（ATG=M）。
> 记忆技巧：密码子表按第一二位碱基分组，同组多为同族氨基酸（如 Gxx ≈ 甘/丙等小氨基酸）。

In [1]:
# 标准遗传密码表（完整 64 项）
BASES = 'TCAG'   # 按行序排表
codontable = {}
aa_order = 'FFLLSSSSYY**CC*W LLLLPPPPHHQQRRRR IIIMTTTTNNKKSSRR VVVVAAAADDEEGGGG'
aa_order = aa_order.replace(' ', '')
k = 0
for b1 in BASES:
    for b2 in BASES:
        for b3 in BASES:
            codontable[b1 + b2 + b3] = aa_order[k]
            k += 1

print('密码子表大小:', len(codontable), '（64 全覆盖）')
print('ATG =', codontable['ATG'], '（起始/甲硫氨酸 M）')
print('TAA =', codontable['TAA'], '（终止 *）')
print('TAG =', codontable['TAG'], ' TGA =', codontable['TGA'])
assert len(codontable) == 64
assert codontable['ATG'] == 'M' and codontable['TAA'] == '*'
print('✅ 完整遗传密码表加载正确')

密码子表大小: 64 （64 全覆盖）
ATG = M （起始/甲硫氨酸 M）
TAA = * （终止 *）
TAG = *  TGA = *
✅ 完整遗传密码表加载正确


## 2️⃣ 6 框翻译

双链 DNA 每条链有 3 个阅读框（从第 0/1/2 位开始），共 **6 框**：
- 正链 3 框：直接按 3 个相位滑窗
- 反链 3 框：先取**反向互补**，再按 3 相位滑窗

In [2]:
COMP = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', 'N': 'N'}

def revcomp(seq):
    return ''.join(COMP.get(b, 'N') for b in reversed(seq.upper()))

def translate(seq):
    seq = seq.upper().rstrip()
    # 去掉不完整尾部
    seq = seq[:len(seq) - len(seq) % 3]
    return ''.join(codontable.get(seq[i:i+3], '?') for i in range(0, len(seq), 3))

dna = 'ATGGCCCAATAA'   # 长度 12，正好 4 密码子
print('DNA :', dna)
print('蛋白:', translate(dna), '（M-A-Q-*，手算 M A Q stop）')
assert translate(dna) == 'MAQ*'
print('✅ 翻译正确（ATG=M, GCC=A, CAA=Q, TAA=*）')

DNA : ATGGCCCAATAA
蛋白: MAQ* （M-A-Q-*，手算 M A Q stop）
✅ 翻译正确（ATG=M, GCC=A, CAA=Q, TAA=*）


## 3️⃣ 手算示例：三段手译

> **① ATG GCT TAA** → 起始 M，GCT=丙氨酸 A，TAA=终止 → **M A ***
> **② ATG GGG** → M G （GGG=甘氨酸 G）
> **③ TAA 开头** → 终止密码子不编码 → 从后续 ATG 才可能开始

> **验证**：GCT 查表 → 第一二位 GC：碱基 G + C → GCC/GCT/GCG/GCA 均为**丙氨酸 A**（同族），
> GGG → 第一二位 GG → GGT/GGC/GGA/GGG = **甘氨酸 G**。

In [3]:
# 验证手算
print('ATG GCT TAA →', translate('ATGGCTTAA'))        # MA*
print('ATG GGG     →', translate('ATGGGG'))           # MG
print('TAA ATG GGG →', translate('TAAATGGGG'))        # *MG（TAA 先出现）
t1 = translate('ATGGCTTAA'); t2 = translate('ATGGGG')
print('GCT 查表 =', codontable['GCT'], ' GGG 查表 =', codontable['GGG'])
assert t1 == 'MA*' and t2 == 'MG' and codontable['GCT'] == 'A' and codontable['GGG'] == 'G'
print('✅ 手译三段全部验证通过')

ATG GCT TAA → MA*
ATG GGG     → MG
TAA ATG GGG → *MG
GCT 查表 = A  GGG 查表 = G
✅ 手译三段全部验证通过


## 4️⃣ ORF 检测

**ORF（开放阅读框）**：从 ATG 起始、到 TAA/TAG/TGA 终止、无中途终止的连续编码区。
> 原核注释常用简化规则：长度 ≥ 阈值（如 300bp）且被 3 整除。
> 6 框全扫，每框找「ATG...终止」的最长片段。

In [4]:
def find_orfs(seq, min_len=6, start='ATG', stops=('TAA', 'TAG', 'TGA')):
    "找 6 框中的 ORF：起始密码子必须与所在 frame 的相位对齐（i % 3 == frame）"
    orfs = []
    seq = seq.upper()
    # 每帧携带相位 p：该 frame 从第 p 位起每 3bp 一组
    frames = [(seq, p) for p in range(3)] + [(revcomp(seq), p) for p in range(3)]
    for s, p in frames:
        for i in range(len(s) - 2):
            if i % 3 != p:          # 相位约束：ATG 必须落在本 frame 的密码子边界
                continue
            if s[i:i+3] == start:
                for j in range(i + 3, len(s) - 2, 3):
                    if s[j:j+3] in stops:
                        if j - i >= min_len * 3:
                            orfs.append(('frame+' + str(p), i, j + 3, s[i:j+3]))
                        break
    return orfs

# 一段含 ATG 开头、后面是完整编码、TAA 终止的序列（5 个编码密码子后终止）
seq = 'TTATCC' + 'ATG' + 'GCA' + 'CAA' + 'ACC' + 'GCT' + 'TTA' + 'TAA' + 'TAGG'
orfs = find_orfs(seq)
print('检测到 ORF 数:', len(orfs))
for f, i, j, s in orfs:
    print('  frame=%s pos=[%d,%d) 长度=%d 翻译=%s' % (f, i, j, len(s), translate(s)))
assert len(orfs) == 1 and translate(orfs[0][3]).endswith('*')
print('✅ ORF 检测完成')

检测到 ORF 数: 1
  frame=frame+0 pos=[6,27) 长度=21 翻译=MAQTAL*
✅ ORF 检测完成


## 5️⃣ GFF 注释文件解析

**GFF3** 每行 9 列（tab 分隔）：
`seqid  source  type  start  end  score  strand  phase  attributes`

| 列 | 含义 |
|---|---|
| seqid | 染色体/contig |
| type | gene / mRNA / CDS / exon ... |
| start, end | 1-based 坐标 |
| strand | + / - |
| attributes | ID=...;Name=...

In [5]:
def parse_gff(text):
    "解析 GFF3 -> [{seqid,type,start,end,strand,attrs}]"
    feats = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        f = line.split('\t')
        if len(f) < 9:
            continue
        feats.append({'seqid': f[0], 'type': f[2], 'start': int(f[3]),
                      'end': int(f[4]), 'strand': f[6], 'attrs': f[8]})
    return feats

gff = '''##gff-version 3
chr1	Augustus	gene	1000	2000	.	+	.	ID=gene1
chr1	Augustus	CDS	1000	1500	.	+	0	Parent=gene1
chr1	Augustus	CDS	1600	2000	.	+	1	Parent=gene1'''
feats = parse_gff(gff)
print('解析特征数:', len(feats))
for ft in feats:
    print('  %s %s %d-%d (%s)' % (ft['seqid'], ft['type'], ft['start'], ft['end'], ft['strand']))
assert len(feats) == 3 and feats[0]['type'] == 'gene'
print('✅ GFF3 解析完成（CDS 由两个 exon 拼接，phase 0/1）')

解析特征数: 3
  chr1 gene 1000-2000 (+)
  chr1 CDS 1000-1500 (+)
  chr1 CDS 1600-2000 (+)
✅ GFF3 解析完成（CDS 由两个 exon 拼接，phase 0/1）


## ✅ 自测清单

- [ ] 能背出 3 个终止密码子与起始密码子
- [ ] 能手写翻译（查表）与 6 框翻译
- [ ] 能实现 ORF 检测（ATG→终止）
- [ ] 能解析 GFF 并解释 CDS/exon 关系